# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amah67/mlintern/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)



## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1: AI Overview Displacement and Traffic Delay

The Finding - Pages experiencing heavy AI overview and generative placement for search have a faster organic click decay rate than those in lower AI overview and generative placement groups.Label Source & Methodology Question: The label is_decaying is based on the comparison between clicks during pre-period (<2026−05−01) and post-period.

Methodology Question - It’s obvious that there is correlation between AI placement and clicks decay rate, but is it enough to claim that these features caused traffic loss? Maybe it’s caused by seasonal changes of search queries or even publisher layout changes?

Finding 2: Striking-Distance CTR Gaps

The Finding - Articles ranked 4-10 that had below average click-through rate (CTR) have a higher probability of disappearing from search.

Label Source & Methodology Question - Calculations performed from pre-period rank position and CTR with respect to traffic loss in the post-period. Are there any survival or selection biases in this approach? In the case that editorial staff decides to measure and optimize those pages that are underperforming, then a portion of the observed effect is human-induced.

In [2]:
import os
import json
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

# 1. Ensure outputs directory exists
os.makedirs("work/outputs", exist_ok=True)
data_path = "work/outputs/features.parquet"

# 2. Resolve dataset source: local cache or direct Hugging Face warehouse query
if os.path.exists(data_path):
    print(f"Loading cached features from {data_path}...")
    df = pd.read_parquet(data_path)
else:
    print("Fetching pre-period signals from Hugging Face warehouse using 'flyrankapi' secret...")
    try:
        hf_token = userdata.get('flyrankapi')
    except Exception as e:
        raise RuntimeError(
            "Secret 'flyrankapi' not found in Colab secrets. "
            "Please click the 🔑 icon on the left panel in Colab, add 'flyrankapi' as the name, "
            "paste your Hugging Face token, and toggle Notebook access ON."
        ) from e

    con = duckdb.connect()
    con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

    rel = "hf://datasets/FlyRank/internship-warehouse"
    dim_content_path = f"read_parquet('{rel}/dim_content.parquet')"
    fact_daily_path = f"read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')"
    dim_clients_path = f"read_parquet('{rel}/dim_clients.parquet')"

    # Clean, robust single-scan query avoiding missing directory paths
    query = f"""
    WITH target_clients AS (
        SELECT client_hash_id
        FROM {dim_clients_path}
        WHERE is_active = TRUE
        LIMIT 25
    ),
    daily_agg AS (
        SELECT
            f.content_hash_id,
            ANY_VALUE(f.client_hash_id) AS client_hash_id,
            AVG(f.gsc_avg_position) FILTER (WHERE f.report_date < DATE '2026-05-01') AS avg_position,
            SUM(f.gsc_clicks) FILTER (WHERE f.report_date < DATE '2026-05-01') AS pre_clicks,
            SUM(f.gsc_impressions) FILTER (WHERE f.report_date < DATE '2026-05-01') AS pre_impressions,
            (SUM(f.sessions_ai) FILTER (WHERE f.report_date < DATE '2026-05-01') * 1.0) /
                NULLIF(SUM(f.ga4_sessions) FILTER (WHERE f.report_date < DATE '2026-05-01'), 0) AS ai_traffic_pct,
            SUM(f.gsc_clicks) FILTER (WHERE f.report_date >= DATE '2026-05-01') AS post_clicks
        FROM {fact_daily_path} f
        JOIN target_clients tc ON f.client_hash_id = tc.client_hash_id
        WHERE f.report_date >= DATE '2026-01-01'
        GROUP BY f.content_hash_id
    )
    SELECT
        d.content_hash_id,
        d.client_hash_id,
        c.content_type,
        c.main_intent,
        c.competition_level,
        c.search_volume,
        c.cpc,
        c.word_count,
        c.char_count,
        c.backlinks,
        d.avg_position,
        COALESCE(d.pre_clicks * 1.0 / NULLIF(d.pre_impressions, 0), 0.0) AS ctr,
        COALESCE(d.ai_traffic_pct, 0.0) AS ai_traffic_pct,
        CASE WHEN COALESCE(d.post_clicks, 0) < COALESCE(d.pre_clicks, 0) THEN 1 ELSE 0 END AS is_decaying
    FROM daily_agg d
    JOIN {dim_content_path} c ON d.content_hash_id = c.content_hash_id
    WHERE c.is_published = TRUE AND c.is_deleted = FALSE;
    """
    df = con.sql(query).df()
    df.to_parquet(data_path, index=False)
    print(f"Cached {len(df):,} rows successfully to {data_path}")

# 3. Define feature columns (exclude IDs, target, and leak sources)
exclude_cols = ["content_hash_id", "client_hash_id", "is_decaying", "post_clicks"]
feature_cols = [c for c in df.columns if c not in exclude_cols]

# Clean feature matrix for training
X = df[feature_cols].select_dtypes(include=[np.number]).fillna(0)
y = df["is_decaying"].astype(int)
groups = df["client_hash_id"]
base_rate = float(y.mean())

print(f"\nFeature matrix shape: {X.shape}")
print(f"Target Base Rate (Decay Rate): {base_rate:.2%}")

Fetching pre-period signals from Hugging Face warehouse using 'flyrankapi' secret...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Cached 129,990 rows successfully to work/outputs/features.parquet

Feature matrix shape: (129990, 8)
Target Base Rate (Decay Rate): 17.29%


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Validation Strategy Audit (Before vs. After):

**Before (Naive Random Split / KFold):** 100.00% Precision@20

Rows from the same client domain appeared in both training and test sets, enabling the model to artificially memorize client-specific signals.

**After (Honest Grouped Split / GroupKFold by client_hash_id):** 85.00% Precision@20

Holding out entire client websites tests true out-of-domain generalization, delivering a 4.9× lift over the 17.29% base rate.

**The Memorization Gap:** 15.00%

This 15-point drop quantifies the exact extent of domain memorization present in naive random splits.

In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import KFold, GroupKFold

exclude_cols = ["content_hash_id", "client_hash_id", "is_decaying", "post_clicks"]
feature_cols = [c for c in df.columns if c not in exclude_cols]

X = df[feature_cols].select_dtypes(include=[np.number]).fillna(0)
y = df["is_decaying"].astype(int)
groups = df["client_hash_id"]

def precision_at_k(scores, labels, k=20):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

# 1. Naive Random Split (Before)
kf = KFold(n_splits=5, shuffle=True, random_state=42)
random_probs = np.zeros(len(df))
for train_idx, val_idx in kf.split(X):
    rf = RandomForestClassifier(n_estimators=50, max_depth=6, random_state=42, n_jobs=-1)
    rf.fit(X.iloc[train_idx], y.iloc[train_idx])
    random_probs[val_idx] = rf.predict_proba(X.iloc[val_idx])[:, 1]

random_p20 = precision_at_k(random_probs, y, k=20)

# 2. Honest Grouped Split (After)
gkf = GroupKFold(n_splits=5)
grouped_probs = np.zeros(len(df))
for train_idx, val_idx in gkf.split(X, y, groups):
    rf = RandomForestClassifier(n_estimators=50, max_depth=6, random_state=42, n_jobs=-1)
    rf.fit(X.iloc[train_idx], y.iloc[train_idx])
    grouped_probs[val_idx] = rf.predict_proba(X.iloc[val_idx])[:, 1]

grouped_p20 = precision_at_k(grouped_probs, y, k=20)

print("=== Split Rigor Comparison Table ===")
print(f"Naive Random Split Precision@20:  {random_p20:.2%}")
print(f"Honest Grouped Split Precision@20: {grouped_p20:.2%}")
print(f"Memorization Gap:           {(random_p20 - grouped_p20):.2%}")
print(f"Base Rate (Naive Floor):    {base_rate:.2%}")

=== Split Rigor Comparison Table ===
Naive Random Split Precision@20:  100.00%
Honest Grouped Split Precision@20: 85.00%
Memorization Gap:           15.00%
Base Rate (Naive Floor):    17.29%


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Leakage Audit on Final Feature Set:

The final honest model relies primarily on pre-period click-through rate (ctr at 77.69% relative importance) and average position (avg_position at 10.52%), which align logically with search decay patterns. Zero future-window or outcome-derived features are present in the feature matrix, and the leakage audit passed successfully.

In [4]:
# 1. Inspect top feature importances from final honest model
final_rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42, n_jobs=-1)
final_rf.fit(X, y)

importances = pd.Series(final_rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("=== Top Feature Importances (Sanity Check) ===")
print(importances.head(8))

# 2. Assert no forbidden leak columns exist
forbidden_terms = ["post_", "future_", "is_decaying", "trend_pct"]
leaks = [col for col in feature_cols if any(term in col for term in forbidden_terms)]
assert len(leaks) == 0, f"Critical Leakage Detected: {leaks}"

print("\n✔ Leakage Audit Passed: Zero future or outcome-derived columns in feature matrix.")



=== Top Feature Importances (Sanity Check) ===
ctr               0.776945
avg_position      0.105154
ai_traffic_pct    0.044764
word_count        0.032789
char_count        0.030747
search_volume     0.005700
cpc               0.002075
backlinks         0.001827
dtype: float64

✔ Leakage Audit Passed: Zero future or outcome-derived columns in feature matrix.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Bold sentence**

Our machine learning model guarantees exact predictions of future traffic collapse, proving that our algorithm successfully stops all content decay before it happens.

**Rewrite in safe language**

Rather than guaranteeing future traffic outcomes, our analysis demonstrates a measured and directional relationship: across our dataset of 129,990 published URLs, pre-period metrics such as click-through rate and average position exhibit a clear observed association with subsequent traffic decline. When evaluated under rigorous 5-fold GroupKFold cross-validation—accounting for a 15.00% memorization gap between naive random splits and domain-isolated evaluation—our Random Forest model ranks high-risk candidate pages at an honest 85.00% Precision@20 (delivering a 4.9× lift over the 17.29% base rate). Rather than automated prevention, this output functions strictly as a decision-support tool to help editorial teams prioritize which pages warrant content refreshes first.